# Multi-frame model (round 2)

**Goal:** find more asteroids by linking more permissively (3 of 5 frames), and test whether a model that looks at the
**whole tracklet**, all frames' stamps along the predicted path, beats tracklet features + XGBoost.

Read `docs/multiframe_model.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import average_precision_score

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.evaluate.choose_threshold import choose_threshold
from asthunt.link.link_detections_into_tracklets import link_detections_into_tracklets
from asthunt.label.label_tracklets import label_tracklets
from asthunt.dataset.build_tracklet_stamp_dataset import build_tracklet_stamp_dataset
from asthunt.features.compute_tracklet_features import FEATURE_COLUMNS
from asthunt.features.measure_forced_photometry import measure_forced_photometry, FORCED_COLUMNS
from asthunt.models.train_multiframe_model import train_multiframe_model
from asthunt.models.predict_multiframe import predict_multiframe
from asthunt.models.train_tracklet_classifier import train_tracklet_classifier
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.models.save_model_bundle import save_model_bundle

project = load_config("project"); paths = project["paths"]
acquire, link, train_settings = load_config("acquire"), load_config("link"), load_config("train")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


nights, field = acquire["nights"], acquire["field"]
device = "mps" if torch.backends.mps.is_available() else "cpu"


def load_detections(night):
    tag = make_unit_tag(unit_of(night))
    detections = pd.read_parquet(paths["processed"] / f"detections__{tag}.parquet")
    detections["score"] = np.load(paths["processed"] / f"realbogus_scores__{tag}.npy")    # stage-1 CNN (out-of-fold for training nights)
    return detections

## 1. Where are v1's missing asteroids lost?
v1 links only tracklets with ≥ 4 of 5 detections, after a CNN cut that keeps 95% of real detections.
Count how many planted fakes each linking setting recovers on one training night.

In [ ]:
validation_detections = pd.concat([load_detections(n) for n in nights["validation"]])
night = nights["train"][1]
detections = load_detections(night)
injections = pd.read_parquet(paths["processed"] / f"injections__{make_unit_tag(unit_of(night))}.parquet")
speed = injections.groupby("injection_id")["rate_arcsec_per_hour"].first()
searchable = set(speed[speed >= link["min_rate_arcsec_per_hour"]].index)

rows = []
for recall, min_detections in [(0.95, 4), (0.95, 3), (0.99, 3)]:
    threshold = choose_threshold(validation_detections["label"], validation_detections["score"], recall)
    kept = detections[detections["score"] >= threshold]
    kept = kept.assign(x=kept["x_common"], y=kept["y_common"])
    tracklets, members = link_detections_into_tracklets(kept, link["pixel_scale_arcsec"], link["min_rate_arcsec_per_hour"],
                                                        link["max_rate_arcsec_per_hour"], link["match_radius_pixels"],
                                                        min_detections, link["max_residual_pixels"])
    tracklets = label_tracklets(tracklets, members, detections)
    linked = set(tracklets.loc[tracklets["label"] == 1, "injection_id"]) & searchable
    rows.append({"stage-1 recall": recall, "min detections": min_detections, "fakes linked": f"{len(linked)}/{len(searchable)}",
                 "bogus tracklets": int((tracklets["label"] == 0).sum())})
pd.DataFrame(rows)

Linking 3 of 5 with a looser stage-1 cut finds ~11% more planted asteroids, and produces ~1,500 chance alignments per
night that a classifier must reject. Those are the **realistic hard negatives** v1 never had.

## 2. Build the tracklet datasets (stamps along every predicted path, in all frames)

In [ ]:
v2_realbogus_threshold = choose_threshold(validation_detections["label"], validation_detections["score"], 0.99)
for split in ["train", "validation"]:
    for n in nights[split]:
        tracklets, _, stamps = build_tracklet_stamp_dataset(unit_of(n), paths, acquire["bad_mask_bits"], link,
                                                            v2_realbogus_threshold, min_detections=3)
        print(f"{split:10s} {n}: {len(tracklets):5d} tracklets ({int(tracklets['label'].sum())} real)")


def load_split(split_nights):
    table = pd.concat([pd.read_parquet(paths["processed"] / f"tracklets_mf__{make_unit_tag(unit_of(n))}.parquet") for n in split_nights], ignore_index=True)
    stamps = np.concatenate([np.load(paths["processed"] / f"tracklet_stamps_mf__{make_unit_tag(unit_of(n))}.npy") for n in split_nights])
    table = pd.concat([table, measure_forced_photometry(stamps)], axis=1)
    return table, stamps

train_table, train_stamps = load_split(nights["train"])
validation_table, validation_stamps = load_split(nights["validation"])

## 3. Look at a 3-detection real tracklet: the signal in the undetected frames

In [ ]:
example = validation_table[(validation_table["label"] == 1) & (validation_table["n_detections"] == 3)].index[0]
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for frame in range(5):
    axes[frame].imshow(validation_stamps[example, frame, 2].astype(float), origin="lower", cmap="gray", vmin=-3, vmax=6)
    axes[frame].set_title(f"frame {frame}"); axes[frame].axis("off")
axes[5].imshow(validation_stamps[example, :, 2].astype(float).mean(axis=0), origin="lower", cmap="gray", vmin=-1.5, vmax=3)
axes[5].set_title("stack (mean)"); axes[5].axis("off")
plt.suptitle(f"difference stamps along the path, mag {validation_table.loc[example, 'mag_mean']:.1f}, stack SNR {validation_table.loc[example, 'stack_snr']:.1f}")
plt.show()

## 4. The benchmark: four ways to score the same tracklets

In [ ]:
y_train, y_validation = train_table["label"].to_numpy(), validation_table["label"].to_numpy()
with_forced = FEATURE_COLUMNS + FORCED_COLUMNS
X = lambda table, columns: np.nan_to_num(table[columns].to_numpy())

model, history = train_multiframe_model(
    {"stamps": train_stamps, "features": X(train_table, FEATURE_COLUMNS), "labels": y_train},
    {"stamps": validation_stamps, "features": X(validation_table, FEATURE_COLUMNS), "labels": y_validation},
    train_settings, device, project["seed"])

scores = {
    "XGBoost, v1 features": train_tracklet_classifier(X(train_table, FEATURE_COLUMNS), y_train).predict_proba(X(validation_table, FEATURE_COLUMNS))[:, 1],
    "XGBoost + forced photometry": train_tracklet_classifier(X(train_table, with_forced), y_train).predict_proba(X(validation_table, with_forced))[:, 1],
    "multi-frame CNN": predict_multiframe(model, validation_stamps, X(validation_table, FEATURE_COLUMNS), device),
}
scores["average of the last two"] = (scores["XGBoost + forced photometry"] + scores["multi-frame CNN"]) / 2

rows = []
for name, s in scores.items():
    row = {"model": name, "AP, all": average_precision_score(y_validation, s)}
    for n in [3, 4, 5]:
        part = (validation_table["n_detections"] == n).to_numpy()
        row[f"AP, {n} detections"] = average_precision_score(y_validation[part], s[part])
    rows.append(row)
benchmark = pd.DataFrame(rows).round(4)
benchmark

In [ ]:
three = (validation_table["n_detections"] == 3).to_numpy()
print(f"3-detection tracklets in validation: {three.sum()}, of which real: {y_validation[three].sum()}")
for column in ["stack_snr", "forced_snr_min"]:
    print(f"{column:15s} median real {validation_table.loc[three & (y_validation == 1), column].median():6.2f}   "
          f"bogus {validation_table.loc[three & (y_validation == 0), column].median():6.2f}")

**Reading the benchmark:**
- All four score ~0.992-0.993 overall. On the hard 3-detection tracklets they differ by a few hundredths, but with only
  ~33 real examples that's within noise.
- Forced photometry is informative on its own (see the medians above), but the existing features (especially the stage-1
  CNN scores) already carry that information.
- **The classifier is not the bottleneck; linking was.** So v2 uses the simplest option that ties: XGBoost on the v1
  features, retrained on the new, harder tracklets.

## 5. Train and save bundle v2

In [ ]:
v2_tracklet_model = train_tracklet_classifier(X(train_table, FEATURE_COLUMNS), y_train, seed=project["seed"])
v2_tracklet_threshold = choose_threshold(y_validation, v2_tracklet_model.predict_proba(X(validation_table, FEATURE_COLUMNS))[:, 1], 0.95)
thresholds = {"realbogus": float(v2_realbogus_threshold), "tracklet": float(v2_tracklet_threshold), "min_detections": 3}
print(thresholds)

v1 = load_model_bundle(paths["models"], "v1")
if not (paths["models"] / "v2").exists():
    save_model_bundle(paths["models"], "v2", v1["realbogus_model"], v2_tracklet_model, thresholds, {
        "description": "v2: v1 CNN as a 99%-recall filter, linking 3 of 5 frames, XGBoost retrained on the harder tracklets",
        "train_nights": nights["train"], "validation_nights": nights["validation"], "quadrants": [[3, 4]],
        "benchmark_validation_ap": benchmark.set_index("model")["AP, all"].to_dict()})
print("bundles:", sorted(p.name for p in paths["models"].iterdir() if p.is_dir()))


## 6. What v2 got wrong, and v4
The scoreboard (next notebook) showed v2 is deeper, but on the dense test night it accepted **15 unknown tracklets**. Vetting by eye:
**all 15 were bogus**, of two kinds:
- **artifact walkers:** slow "tracks" stepping along a faint line or star residual present in every frame;
- **noise alignments:** three ~2σ blips on a line, with nothing at the predicted spot in the other frames.

Two physics features target exactly those (`features/add_path_features.py`):
- `static_neighbours_*`: detections in *other* frames at the *same* sky position (movers leave nothing behind);
- forced photometry along the path (`stack_snr`, `forced_snr_min`, …): is there signal where the line says it should be?

On the test night: known asteroids had static 0.0 / forced-min 6.4; the 15 bogus candidates 1.0 / −0.4.
*(Honesty note: the test night was used to find these failure modes, so from now on the **holdout** night is the clean judge.)*


In [ ]:

from asthunt.features.add_path_features import add_path_features, ALL_FEATURE_COLUMNS


def load_split_with_path_features(split_nights):
    tables, stamps_list = [], []
    for n in split_nights:
        tag = make_unit_tag(unit_of(n))
        table = pd.read_parquet(paths["processed"] / f"tracklets_mf__{tag}.parquet")
        members = pd.read_parquet(paths["processed"] / f"tracklet_members_mf__{tag}.parquet")
        stamps = np.load(paths["processed"] / f"tracklet_stamps_mf__{tag}.npy")
        tables.append(add_path_features(table, members, load_detections(n), stamps.astype(np.float32)))
        stamps_list.append(stamps)
    return pd.concat(tables, ignore_index=True), np.concatenate(stamps_list)

train_v3, _ = load_split_with_path_features(nights["train"])
validation_v3, _ = load_split_with_path_features(nights["validation"])
v3_model = train_tracklet_classifier(X(train_v3, ALL_FEATURE_COLUMNS), train_v3["label"].to_numpy(), seed=project["seed"])
v3_scores = v3_model.predict_proba(X(validation_v3, ALL_FEATURE_COLUMNS))[:, 1]
y3 = validation_v3["label"].to_numpy()
print(f"validation AP v4: {average_precision_score(y3, v3_scores):.4f}")
for n in [3, 4]:
    part = (validation_v3["n_detections"] == n).to_numpy()
    print(f"  {n} detections: AP {average_precision_score(y3[part], v3_scores[part]):.3f}")
importance = pd.Series(v3_model.feature_importances_, index=ALL_FEATURE_COLUMNS).sort_values()
importance.plot.barh(figsize=(6, 5), title="v4 feature importance"); plt.show()

real = train_v3[train_v3["label"] == 1]
print(f"real tracklets with static_neighbours_mean >= 1: {(real['static_neighbours_mean'] >= 1).mean():.1%}  (the buggy first version: 22%)")


In [ ]:

v3_threshold = choose_threshold(y3, v3_scores, 0.95)
v3_thresholds = {"realbogus": float(v2_realbogus_threshold), "tracklet": float(v3_threshold), "min_detections": 3}
print(v3_thresholds)
# (A first "v3" was trained while count_static_neighbours still counted a tracklet's OWN detections:
#  22% of real tracklets were flagged. It's retired in data/superseded/retired_bundles/v3. This is v4.)
if not (paths["models"] / "v4").exists():
    save_model_bundle(paths["models"], "v4", v1["realbogus_model"], v3_model, v3_thresholds, {
        "description": "v4: v2 + static-neighbour and forced-photometry features (17 features)",
        "feature_columns": ALL_FEATURE_COLUMNS,
        "train_nights": nights["train"], "validation_nights": nights["validation"], "quadrants": [[3, 4]]})
print("bundles:", sorted(p.name for p in paths["models"].iterdir() if p.is_dir()))


Now run `notebooks/scoreboard.ipynb`: it evaluates every bundle on validation, test and holdout.

## Exercises
1. Change `min_detections` to 2. What happens to the number of bogus tracklets? Why is 2 points not enough to call something a track?
2. Train the multi-frame CNN without the `features` input (stamps only). How much of its score came from the numbers?
3. Use `stack_snr` alone as a score. What AP does pure physics get?